Visualise model output from CF registry data vs baseline ppFEV1

In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import cfr.cfr_viz_helpers as vh

In [ ]:
# Load AC with inferred from 2023 data, 2nd day = 2019 data

df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)

# EXCEL
# 2 entries means there is a 2019 entry for every 2023 entry
# df_res = bd.load_meas_from_excel(
#     "infer_AR_using_19_23_data_2entries_fev1_10122025",
#     # "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
#     study_folder="CFR",
#     str_cols_to_arrays=["Airway resistance (%)"],
# )
# df_res = df_res.drop(columns=["Healthy FEV1 (L)"])

# CSV
# df_res = (
#     bd.load_meas_from_excel(
#         # "infer_AR_using_two_days_model_19_23_data_2entries_fev1_10122025",
#         "infer_AR_using_two_days_model_19_23_data_2entries_fev1_fef2575_10122025",
#         study_folder="CFR",
#         str_cols_to_arrays=["Airway resistance (%)"],
#         use_csv=True,
#         date_cols=["Day"],
#         bypass_sanity_checks=True,
#     )
#     .drop(columns=["Healthy FEV1 (L)"])
#     .rename(columns={"Day": "Date Recorded"})
# )

# Merging
# df = df_res.merge(df_meas, on=["ID", "Date Recorded"])

In [2]:
# Load AC from 2019 data with 2nd day = best FEV1 (no FEF2575)
# df = bd.load_meas_from_excel("AR_19_data_with_best_FEV1", study_folder="CFR", str_cols_to_arrays=["Airway resistance (%)"])
df = bd.load_meas_from_excel(
    "infer_all_19_data_with_best_FEV1",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(HFEV1|FEV1)",
        "P(HFEV1|bFEV1, FEV1)",
        "P(HFEV1|FEF2575, bFEV1, FEV1)",
    ],
)
print(f"Shape: {df.shape}")

Shape: (2037, 22)


In [29]:
# Process

# Keep only values from 2023
# df23 = df[df["Date Recorded"] == datetime.date(2023, 1, 1)]

df23 = df

AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df23[AC.name] = df23[AR.name].apply(lambda arr: arr[::-1])
df23["ecFEV1 % Predicted (clipped)"] = df23["ecFEV1 % Predicted"].clip(upper=100)
df23["P(ppFEV1|AC)"] = vh.calc_P_ppFEV1_given_AC(df23, AC)
df["P(ppFEV1|AC) ratioed"] = vh.calc_P_ppFEV1_given_AC(df, AC, corr=True)

# Airway conductance

In [ ]:
## FILL ##
ratioed = False
prctile = 25
prctile = 100

df_to_plot, t = vh.filter_confidently_disagreeing_examples(df, prctile, ratioed)
# df_to_plot = df23[df23["P(ppFEV1|AC)"] <= t]

# title = f"Dumbell plot for CF Registry 2023, 2019 2nd day, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, 2019 2nd day, FEV1 & FEF25-75 (2entries), {t*100:.2f}% conf. disagreeing"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1 (no FEF25-75)"
title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 & FEF2575 (2entries)"

ac_col = AC.name

fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

# ppfev1_row = "ecFEV1 % Predicted (clipped)"
ppfev1_row = "ecFEV1 % Predicted"

df_to_plot, _, _ = vh.get_dumbell_plot_data(
    df_to_plot, AC.name, AC, ppfev1_row=ppfev1_row
)

# Split dataframe between mild, moderate and severe CF lung disease
# Equivalent to Mean AR_ecFEV1% < 30%, 30 to 60 and > 60%
# Get unique IDs and their corresponding Mean AR_ecFEV1% values
mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
# mask_ppfev1 = df_to_plot["measure"] == f"Mean {ac_col} prediction"

id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)})"

# Print the sizes to verify
print(f"# Mild: {len(mild_ids)}")
print(f"# Moderate: {len(moderate_ids)}")
print(f"# Severe: {len(severe_ids)}")

# Plot the three groups
vh.plot_dumbell_for_df(
    fig, df_mild, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 3
)
vh.plot_dumbell_for_df(
    fig, df_moderate, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 2
)
vh.plot_dumbell_for_df(
    fig, df_severe, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 1
)


fig.update_layout(
    height=1000,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="Airway conductance (%)",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(
    # f"{dh.get_path_to_main()}PlotsBreathe/Dumbell_plot_AR_ecFEV1_by_severity/{title}_clipped.pdf"
    # f"{dh.get_path_to_main()}PlotsCFR/{title}_clipped.pdf"
    f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf"
)
# fig.show()

# On this plot, if a person is very sick, it's best fev1 measurement will contain a lot of inflammatory markers (sputum, airway wall inflammation).
# The sicker the person,the more underestimated the AR pred is because the maximum FEV1 blown is not healthy.
# Let's add the FEF25-75.

# 60% of data falls within red band

# TODO: add vertical lines corresponding to key AR values

# Longitudinal AR profile on the web app

# Mild: 890
# Moderate: 793
# Severe: 354
11.988144416876157


# FEV1 % personalised predicted (Non saturating)

In [ ]:
# df = bd.load_meas_from_excel(
#     "pppfev1_ppfev1_IV_19_assoc",
#     study_folder="CFR",
#     str_cols_to_arrays=[
#         "Airway resistance (%)",
#         "P(HFEV1|FEF2575, bFEV1, FEV1)",
#         "P(HFEV1|FEV1)",
#     ],
# )

In [6]:
HFEV1 = mh.VariableNode("Healthy FEV1 (L)", 1, 6, 0.05, prior=None)

df["mean FEV1PersPred"] = df["P(HFEV1|FEF2575, bFEV1, FEV1)"].apply(HFEV1.get_mean)
df["mean FEV1PredFT"] = df["P(HFEV1|bFEV1, FEV1)"].apply(HFEV1.get_mean)
df["mean FEV1PredST"] = df["P(HFEV1|FEV1)"].apply(HFEV1.get_mean)
df["FEV1%PersPred"] = df["FEV1"] / df["mean FEV1PersPred"] * 100
df["FEV1%PredFT"] = df["FEV1"] / df["mean FEV1PredFT"] * 100
df["FEV1%PredST"] = df["FEV1"] / df["mean FEV1PredST"] * 100

In [7]:
df["pppFEV1 - ppFEV1ST"] = df["FEV1%PersPred"] - df["FEV1%PredST"]
df["ppFEV1FT - ppFEV1ST"] = df["FEV1%PredFT"] - df["FEV1%PredST"]
df.sort_values(by=["pppFEV1 - ppFEV1ST"], inplace=True)

In [9]:
prctile = 50
# prctile = 0

pppFEV1_col = 'FEV1%PersPred'
pppFEV1_col = 'FEV1%PredFT'

title = f"Dumbell plot for 2019 CFR data FEV1%PredST vs {pppFEV1_col} {prctile:.0f}th prctile"

diff_col = "pppFEV1 - ppFEV1ST"
t = df[diff_col].abs().quantile(prctile / 100)
df_to_plot = df[df[diff_col].abs() > t]

fig = make_subplots(
    1, 3, horizontal_spacing=0.2, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

# Create the three dataframe
mask_mild = df_to_plot["FEV1%PredST"] >= 70
mask_moderate = (df_to_plot["FEV1%PredST"] >= 40) & (df_to_plot["FEV1%PredST"] < 70)
mask_severe = 40 > df_to_plot["FEV1%PredST"]

vh.plot_scalar_dumbell(
    fig, df_to_plot[mask_severe], "FEV1%PredST", pppFEV1_col, col=1
)
vh.plot_scalar_dumbell(
    fig, df_to_plot[mask_moderate], "FEV1%PredST", pppFEV1_col, col=2
)
vh.plot_scalar_dumbell(
    fig, df_to_plot[mask_mild], "FEV1%PredST", pppFEV1_col, col=3
)

fig.update_layout(
    height=1400 if prctile == 0 else 800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(
    range=[-1, 101],
    tickvals=[0, 40, 70, 100],
    title="FEV1 % predicted",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
# fig.show()

In [ ]:
prctile = 50
# prctile = 0

title = f"Dumbell plot for 2019 CFR data with best FEV1, {prctile:.0f}th prctile"

diff_col = "pppFEV1 - ppFEV1ST"
t = df[diff_col].abs().quantile(prctile / 100)
df_to_plot = df[df[diff_col].abs() > t]

fig = make_subplots(
    1, 3, horizontal_spacing=0.2, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

# Create the three dataframe
mask_mild = df_to_plot["FEV1%PredST"] >= 70
mask_moderate = (df_to_plot["FEV1%PredST"] >= 40) & (df_to_plot["FEV1%PredST"] < 70)
mask_severe = 40 > df_to_plot["FEV1%PredST"]

vh.plot_scalar_dumbell(
    fig, df_to_plot[mask_severe], "FEV1%PredST", "FEV1%PersPred", col=1
)
vh.plot_scalar_dumbell(
    fig, df_to_plot[mask_moderate], "FEV1%PredST", "FEV1%PersPred", col=2
)
vh.plot_scalar_dumbell(
    fig, df_to_plot[mask_mild], "FEV1%PredST", "FEV1%PersPred", col=3
)

fig.update_layout(
    height=1400 if prctile == 0 else 800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(
    range=[-1, 101],
    tickvals=[0, 40, 70, 100],
    title="FEV1 % predicted",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
# fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
fig.show()

### Compare FEV1%PredST with FEV1%Pred

In [13]:
df.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'best FEV1', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted',
       'best FEV1 old', 'idx FEV1', 'idx FEF2575%FEV1', 'idx best FEV1',
       'P(HFEV1|FEF2575, bFEV1, FEV1)', 'P(HFEV1|FEV1)',
       'Airway resistance (%)', 'Hosp IVs', 'Hosp IV days', 'Home IVs',
       'Home IV days', 'Oral', 'Non Hosp IVs', 'Non Hosp IV days',
       'Chest episodes', 'Cough episodes', 'Pulm Abscess', 'Smoking status',
       '2nd hand smoking exposure', 'IVs', 'IV days', 'mean HFEV1_pers',
       'mean HFEV1_ST', 'FEV1%PersPred', 'FEV1%PredST', 'pppFEV1 - ppFEV1ST'],
      dtype='object')

In [24]:
import plotly.graph_objects as go

title = f"Dumbell plot for 2019 CFR data with bFEV1, FEV1%Pred vs FEV1%PredST"
title = f"Dumbell plot for 2019 CFR data with bFEV1, FEV1%PredST vs FEV1%PersPred"

df["baseline diff"] = df["FEV1%PredST"] - df["FEV1 % Predicted"]
df.sort_values(by="baseline diff", inplace=True)
df.sort_values(by="pppFEV1 - ppFEV1ST", ascending=False, inplace=True)
# diff_col = "pppFEV1 - ppFEV1ST"
# t = df[diff_col].abs().quantile(prctile / 100)
# df_to_plot = df[df[diff_col].abs() > t]

df_to_plot = df[df["FEV1 % Predicted"] >= 100]
df_to_plot = df

fig = make_subplots(1, 1)

baseline_col = "FEV1%PredST"
baseline_col = "FEV1%PredST"
model_col = "FEV1 % Predicted" 
model_col = "FEV1%PersPred" 
fig.add_trace(
    go.Scatter(
        x=df_to_plot[model_col],
        y=df_to_plot["ID"],
        mode="markers",
        # marker=dict(color="lightblue", size=4),
        marker=dict(color="red", size=4),
        name=model_col,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=df_to_plot[baseline_col],
        y=df_to_plot["ID"],
        mode="markers",
        marker=dict(color="blue", size=4),
        name=baseline_col,
    ),
    row=1,
    col=1,
)

fig.update_layout(
    height=1400 if prctile == 0 else 800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(
    range=[-1, 130],
    # tickvals=[0, 40, 70, 100],
    title="FEV1 % predicted",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
# fig.show()

## Old version

In [ ]:
## FILL ##
prctile = 75
prctile = 0

title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, FEV1%PersPred {prctile:.0f}th prctile"
ac_col = "FEV1%PersPred"
ppfev1_row = "ecFEV1 % Predicted"

df["ppFEV1 - pppFEV1"] = df["FEV1 % Predicted"] - df["FEV1%PersPred"]
col = "ppFEV1 - pppFEV1"
t = df[col].abs().quantile(prctile / 100)
df_to_plot = df[df[col].abs() > t]


df_to_plot, _, _ = get_dumbell_plot_data_model_ppfev1(df_to_plot, ac_col, ppfev1_row)

mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

# Plot the three groups
fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)
title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)})"
plot_dumbell_for_df_model_ppfev1(fig, df_mild, [ac_col, ppfev1_row], 3)
plot_dumbell_for_df_model_ppfev1(fig, df_moderate, [ac_col, ppfev1_row], 2)
plot_dumbell_for_df_model_ppfev1(fig, df_severe, [ac_col, ppfev1_row], 1)

fig.update_layout(
    # height=1000,
    height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="FEV1 % predicted",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
# fig.show()

13.3130788065074


# Does rank change between AC and FEV1%PersPred?

In [110]:
df["AC mean"] = df[AC.name].apply(AC.get_mean)
df["ppFEV1 - AC mean"] = df["ecFEV1 % Predicted"] - df["AC mean"]

In [ ]:
# Issue: pppFEV1 is based on AR being 0-2%, not 0%!! →  means pppFEV1 might be lower than AC mean. Hence diff < 0

# How does ppFEV1 metrics compare to AC mean?
df["diff pppFEV1 - diff AC mean"] = df["ppFEV1 - AC mean"] - df["ppFEV1 - pppFEV1"]

In [ ]:
df = df.sort_values(by=["diff pppFEV1 - diff AC mean"], ascending=False)

In [112]:
df[df["diff pppFEV1 - diff AC mean"].abs() > 6][
    [
        "ID",
        "Age",
        "Height",
        "Sex",
        "FEV1",
        "FEF2575",
        "best FEV1",
        "FEV1 % Predicted",
        "FEV1%PersPred",
        "AC mean",
        "diff pppFEV1 - diff AC mean",
        "ppFEV1 - AC mean",
        "ppFEV1 - pppFEV1",
    ]
]

,ID,Age,Height,Sex,FEV1,FEF2575,best FEV1,FEV1 % Predicted,FEV1%PersPred,AC mean,diff pppFEV1 - diff AC mean,ppFEV1 - AC mean,ppFEV1 - pppFEV1
2033,C222739,69,153,Female,1.63,0.66,1.63,82.817413,81.761685,74.957455,6.804230,7.859958,1.055727
1540,B165718,57,162,Female,2.17,0.76,2.17,84.694427,83.616476,77.105186,6.511290,7.589241,1.077951
1950,B172185,73,176,Male,2.34,1.02,2.34,76.985285,77.543842,71.314175,6.229667,5.671110,-0.558557
1485,B164954,52,172,Female,2.68,0.97,2.73,87.582749,85.300512,79.298046,6.002467,8.284703,2.282236
279,B157577,21,159,Female,2.45,2.98,2.45,77.200572,78.376171,84.452097,-6.075926,-7.251525,-1.175599
1624,B166717,29,162,Female,1.69,1.72,1.80,52.755536,54.236918,60.348532,-6.111614,-7.592996,-1.481382
998,B162253,20,158,Female,2.60,4.05,2.60,82.937739,83.081858,89.569108,-6.487250,-6.631369,-0.144119
176,B156968,29,159,Female,2.32,2.96,2.41,75.350849,76.373233,83.031852,-6.658619,-7.681003,-1.022384
1600,B166359,20,162,Female,2.41,2.89,2.47,72.906318,74.362433,81.026436,-6.664003,-8.120118,-1.456114
1974,C219729,21,168,Female,2.11,2.25,2.17,59.158449,60.782046,67.516635,-6.734589,-8.358186,-1.623597


In [ ]:
import src.cfr.cfr_helpers as cfrh


hfev1_inf = df.loc[df["ID"] == "B156254"].apply(cfrh.infer_hfev1, axis=1).values[0]

In [ ]:
import inference.helpers as ih
from plotly.subplots import make_subplots

row = df.loc[df["ID"] == "B156254"].iloc[0]

fig = make_subplots(rows=3, cols=1)

ih.plot_histogram(fig, AC, row[AC.name], AC.a, AC.b, 1, 1, name=AC.name)
ih.plot_histogram(
    fig,
    ecFEV1,
    row["P(FEV1|HFEV1_pers)"],
    ecFEV1.a,
    ecFEV1.b,
    2,
    1,
    name="P(FEV1|HFEV1_pers)",
)
fig.add_vline(
    x=row["FEV1%PersPred"],
    line_width=2,
    line_color="black",
    row=1,
    col=1,
    name="FEV1%PersPred",
)
ih.plot_histogram(
    fig,
    HFEV1,
    row["P(HFEV1|FEF2575, bFEV1)"],
    0,
    HFEV1.b,
    3,
    1,
    name="P(HFEV1|FEF2575, bFEV1)",
)

fig.update_layout(
    title=f"ID {row['ID']}: {AC.name} and P(FEV1|HFEV1_pers) overlay",
    barmode="overlay",
    width=800,
    height=400,
)
fig.show()

In [167]:
import numpy as np
from pgmpy.inference.ExactInference import VariableElimination

import models.builders as mb

row = df.loc[df["ID"] == "B156254"].iloc[0]

id, height, age, sex = row[["ID", "Height", "Age", "Sex"]]
# ar_prior = "breathe (2 days model, ecFEV1 addmultnoise, ecFEF25-75)"
ar_prior = "uniform"
ecfev1_noise_model_suffix = "_std_add_mult_ecfev1"
fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"

(
    model,
    HFEV1,
    AR_vars,
    uFEV1_vars,
    ecFEV1_vars,
    ecFEF2575prctecFEV1_vars,
) = mb.fev1_fef2575_n_day_BN_noise(
    2,
    height,
    age,
    sex,
    ar_prior,
    fef2575_cpt_suffix,
    ecfev1_noise_model_suffix,
)
var_elim = VariableElimination(model)

evidence_dict = {}
evidence_dict[ecFEV1_vars[1].name] = row["idx best FEV1"]
evidence_dict[ecFEV1_vars[0].name] = row["idx FEV1"]
evidence_dict[ecFEF2575prctecFEV1_vars[0].name] = row["idx FEF2575%FEV1"]
# evidence_dict[ecFEF2575prctecFEV1_vars[0].name] = 0

res_ve = var_elim.query(
    variables=[HFEV1.name],
    evidence=evidence_dict,
    joint=False,
)
dist_hfev1 = res_ve[HFEV1.name].values

mean_hfev1 = HFEV1.get_mean(dist_hfev1)
pppfev1 = row["FEV1"] / mean_hfev1 * 100

# PLOT
fig = make_subplots(rows=3, cols=1)

ih.plot_histogram(fig, AC, row[AC.name], AC.a, AC.b, 1, 1, name=AC.name)
ih.plot_histogram(
    fig,
    ecFEV1,
    row["P(FEV1|HFEV1_pers)"],
    ecFEV1.a,
    ecFEV1.b,
    2,
    1,
    name="P(FEV1|HFEV1_pers)",
)
fig.add_vline(
    x=pppfev1,
    line_width=2,
    line_color="black",
    row=1,
    col=1,
    name="FEV1%PersPred",
)
ih.plot_histogram(
    fig,
    HFEV1,
    dist_hfev1,
    0,
    HFEV1.b,
    3,
    1,
    name="P(HFEV1|FEF2575, bFEV1)",
)

fig.update_layout(
    title=f"ID {row['ID']}: {AC.name} and P(FEV1|HFEV1_pers) overlay",
    barmode="overlay",
    width=800,
    height=400,
)
fig.show()

In [ ]:
import numpy as np
from pgmpy.inference.ExactInference import VariableElimination

import models.builders as mb

id, height, age, sex = row[["ID", "Height", "Age", "Sex"]]
# ar_prior = "breathe (2 days model, ecFEV1 addmultnoise, ecFEF25-75)"
ar_prior = "uniform"
ecfev1_noise_model_suffix = "_std_add_mult_ecfev1"
fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"

(
    model,
    HFEV1,
    AR,
    uFEV1,
    ecFEV1,
    ecFEF2575prctecFEV1,
) = mb.fev1_fef2575_1_day_BN_noise(
    height,
    age,
    sex,
    ar_prior,
    fef2575_cpt_suffix,
    ecfev1_noise_model_suffix,
)

var_elim = VariableElimination(model)

evidence_dict = {}
evidence_dict[ecFEV1.name] = row["idx best FEV1"]
evidence_dict[ecFEF2575prctecFEV1.name] = row["idx FEF2575%FEV1"]
# evidence_dict[ecFEF2575prctecFEV1.name] = 0

res_ve = var_elim.query(
    variables=[HFEV1.name],
    evidence=evidence_dict,
    joint=False,
)
dist_hfev1 = res_ve[HFEV1.name].values

In [ ]:
# df.to_excel(
#     dh.get_path_to_main() + "ExcelFiles/CFR/dumbel_plots.xlsx",
#     index=False,
# )